# Embedded Poet

## Set up environment

In [1]:
# --- Setup: install-if-missing and NLTK resources ---
import sys, importlib, subprocess

def ensure_packages(pkgs):
    for name in pkgs:
        modname = "sentence_transformers" if name == "sentence-transformers" else name
        try:
            importlib.import_module(modname)
        except ImportError:
            subprocess.check_call([sys.executable, "-m", "pip", "install", name, "-q"])

ensure_packages(["pandas","nltk","scikit-learn"])

# NLTK resources
import nltk
def ensure_nltk(res, path):
    try:
        nltk.data.find(path)
    except LookupError:
        nltk.download(res, quiet=True)

ensure_nltk("stopwords", "corpora/stopwords")
ensure_nltk("punkt", "tokenizers/punkt")
# punkt_tab exists in newer NLTK; safe to try
try:
    nltk.data.find("tokenizers/punkt_tab")
except LookupError:
    try: nltk.download("punkt_tab", quiet=True)
    except Exception: pass

print("✅ Environment ready")



✅ Environment ready


In [2]:
# --- Load prepared corpus (CSV from Task 1) ---
from pathlib import Path
import pandas as pd

CSV_IN = Path("dickinson_prepared_utf8.csv")  # adjust if needed

if not CSV_IN.exists():
    raise FileNotFoundError(f"Can't find {CSV_IN.resolve()} — set CSV_IN to your prepared file")

df = pd.read_csv(CSV_IN, encoding="utf-8")  # expects id,title,text
assert {"id","title","text"}.issubset(df.columns), df.columns

print(f"Loaded poems: {len(df)}")
df.head(3)


Loaded poems: 448


,id,title,text
0,1,SUCCESS.,\nSuccess is counted sweetest\nBy those who ne...
1,2,"Our share of night to bear,","Our share of night to bear,\nOur share of morn..."
2,3,ROUGE ET NOIR.,"\nSoul, wilt thou toss again?\nBy just such a ..."


In [3]:
# --- Tokenizer (stopwords removed, keep apostrophes) ---
import re
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

STOPS = set(stopwords.words("english"))
STOPS.update({"'s", "'t", '"', "'", "n't", "'m", "'ve", "'ll", "'d"})
WORD_RE = re.compile(r"[A-Za-z’']+")

def tokenize_no_stop(text: str):
    toks = [t.lower() for t in word_tokenize(str(text)) if WORD_RE.fullmatch(t)]
    return [t for t in toks if t not in STOPS]

docs_tokens = [tokenize_no_stop(t) for t in df["text"]]
docs_joined = [" ".join(toks) for toks in docs_tokens]

avg_len = sum(map(len, docs_tokens))/max(1,len(docs_tokens))
print("Tokenized docs:", len(docs_tokens), "| avg tokens/poem:", round(avg_len, 2))


Tokenized docs: 448 | avg tokens/poem: 36.42


## Thematic exploration

### Topic modeling

In [4]:
# --- LDA topics (unigrams + bigrams) ---
import csv
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation

K_TOPICS   = 12     # tweak 8–20
TOPN_WORDS = 15
MIN_DF     = 2

cv = CountVectorizer(
    token_pattern=r"[A-Za-z’']+",
    lowercase=True,
    stop_words=None,    # already removed
    ngram_range=(1,2),
    min_df=MIN_DF
)
Xc = cv.fit_transform(docs_joined)
vocab = cv.get_feature_names_out()

lda = LatentDirichletAllocation(
    n_components=K_TOPICS,
    random_state=42,
    learning_method="batch"
)
W = lda.fit_transform(Xc)  # doc-topic
H = lda.components_        # topic-term

# Top words per topic
topic_rows = []
for k in range(K_TOPICS):
    idx = H[k].argsort()[::-1][:TOPN_WORDS]
    words = [vocab[i] for i in idx]
    weights = [float(H[k, i]) for i in idx]
    topic_rows.append({"topic": k, "top_words": ", ".join(words), "weights": ", ".join(f"{w:.2f}" for w in weights)})
topics_df = pd.DataFrame(topic_rows)
topics_df.to_csv("topics_top_words.csv", index=False, encoding="utf-8", lineterminator="\n", quoting=csv.QUOTE_MINIMAL)

# Doc-topic weights (long form)
doc_topic_rows = []
for i, (pid, title) in enumerate(zip(df["id"], df["title"])):
    for k in range(K_TOPICS):
        doc_topic_rows.append({"id": int(pid), "title": str(title), "topic": k, "weight": float(W[i, k])})
doc_topic_df = pd.DataFrame(doc_topic_rows)
doc_topic_df.to_csv("doc_topic_weights.csv", index=False, encoding="utf-8", lineterminator="\n", quoting=csv.QUOTE_MINIMAL)

print("Saved: topics_top_words.csv, doc_topic_weights.csv")
topics_df.head(3)


Saved: topics_top_words.csv, doc_topic_weights.csv


,topic,top_words,weights
0,0,"'', one, tell, emily, dickinson, emily dickins...","37.06, 18.00, 12.28, 12.08, 12.08, 12.08, 10.0..."
1,1,"little, life, like, see, one, would, sun, morn...","18.17, 17.58, 12.00, 11.80, 11.65, 10.16, 9.66..."
2,2,"could, one, see, thee, day, new, would, sing, ...","7.87, 7.67, 6.87, 5.69, 5.51, 5.23, 5.14, 4.81..."


In [11]:
# --- Quick peeks ---
print("Topics sample:")
display(topics_df.head(10))

Topics sample:


,topic,top_words,weights
0,0,"'', one, tell, emily, dickinson, emily dickins...","37.06, 18.00, 12.28, 12.08, 12.08, 12.08, 10.0..."
1,1,"little, life, like, see, one, would, sun, morn...","18.17, 17.58, 12.00, 11.80, 11.65, 10.16, 9.66..."
2,2,"could, one, see, thee, day, new, would, sing, ...","7.87, 7.67, 6.87, 5.69, 5.51, 5.23, 5.14, 4.81..."
3,3,"like, tell, many, bee, done, rose, must, away,...","13.26, 9.19, 7.70, 6.42, 6.33, 5.97, 5.90, 5.6..."
4,4,"little, day, like, know, till, us, could, one,...","22.73, 22.61, 20.75, 18.69, 17.37, 13.06, 11.5..."
5,5,"little, one, away, like, heaven, never, could,...","22.65, 17.64, 15.19, 13.75, 12.89, 12.75, 11.5..."
6,6,"like, summer, know, one, still, god, death, ne...","18.04, 13.33, 12.71, 10.06, 9.47, 8.97, 8.53, ..."
7,7,"never, upon, go, except, could, heaven, must, ...","13.41, 9.85, 9.78, 9.26, 9.22, 8.56, 7.97, 7.5..."
8,8,"upon, like, one, could, '', look, still, know,...","20.21, 19.33, 13.89, 10.32, 9.00, 8.95, 8.33, ..."
9,9,"like, one, little, summer, upon, sun, could, b...","16.13, 15.86, 12.60, 12.27, 11.23, 10.40, 10.2..."


In [13]:
# --- Label topics by inspecting top words and the most representative poems ---

import numpy as np
import pandas as pd

TOPN_WORDS = 12
TOPN_DOCS  = 5

def top_terms_for_topic(k, topn=TOPN_WORDS):
    idx = H[k].argsort()[::-1][:topn]
    return [vocab[i] for i in idx], idx

def top_docs_for_topic(k, topn=TOPN_DOCS):
    weights = W[:, k]
    top_idx = np.argsort(weights)[::-1][:topn]
    rows = []
    for i in top_idx:
        poem_id  = int(df.iloc[i]["id"])
        title    = str(df.iloc[i]["title"])
        weight   = float(weights[i])
        # short excerpt: first 2 non-empty lines
        lines = [ln for ln in str(df.iloc[i]["text"]).splitlines() if ln.strip()][:2]
        excerpt = " / ".join(lines)
        rows.append({"id": poem_id, "title": title, "topic": k, "weight": round(weight, 3), "excerpt": excerpt})
    return pd.DataFrame(rows)

# Example: inspect topics 0..2
for k in range(min(3, lda.n_components)):
    words, _ = top_terms_for_topic(k)
    print(f"\nTopic {k}: {', '.join(words)}")
    display(top_docs_for_topic(k))



Topic 0: '', one, tell, emily, dickinson, emily dickinson, poems, verses, first, know, time, letter


,id,title,topic,weight,excerpt
0,115,"I never lost as much but twice,",0,0.998,"I never lost as much but twice, / And that was..."
1,198,THE SLEEPING FLOWERS.,0,0.988,"""Whose are the little beds,"" I asked, / ""Which..."
2,39,RENUNCIATION.,0,0.988,First page of Renunciation Second page of Renu...
3,177,THE LETTER.,0,0.987,"""Going to him! Happy letter! Tell him â / Te..."
4,197,APRIL.,0,0.980,An altered look about the hills; / A Tyrian li...



Topic 1: little, life, like, see, one, would, sun, morning, eyes, thing, go, could


,id,title,topic,weight,excerpt
0,263,GOING.,1,0.986,"On such a night, or such a night, / Would anyb..."
1,58,PURPLE CLOVER.,1,0.986,"There is a flower that bees prefer, / And butt..."
2,180,AT HOME.,1,0.984,"The night was wide, and furnished scant / With..."
3,388,THE COMING OF NIGHT.,1,0.982,"How the old mountains drip with sunset, / A..."
4,193,THE SUN'S WOOING.,1,0.979,The sun just touched the morning; / The mornin...



Topic 2: could, one, see, thee, day, new, would, sing, heaven, face, sea, till


,id,title,topic,weight,excerpt
0,139,TOO MUCH.,2,0.983,"I should have been too glad, I see, / Too lift..."
1,259,FOLLOWING.,2,0.978,"I had no cause to be awake, / My best was gone..."
2,213,THE MUSHROOM.,2,0.974,"The mushroom is the elf of plants, / At evenin..."
3,200,THE ORIOLE'S SECRET.,2,0.973,"To hear an oriole sing / May be a common thing,"
4,127,THE MARTYRS.,2,0.972,Through the straight pass of suffering / The m...


In [14]:
# labeling topic 0

# --- Collect poems whose dominant topic is 0; add weight & excerpt; save ---
from pathlib import Path

# Try to use in-memory W (doc×topic); else load the long-form CSV
if "W" in globals():
    dom_idx = W.argmax(axis=1)
    dom_weight = W.max(axis=1)
    mask = (dom_idx == 0)
    topic0 = df.loc[mask, ["id","title","text"]].copy()
    topic0["weight"] = dom_weight[mask]
else:
    # Load long-form doc_topic_weights.csv (from Task 5)
    long = pd.read_csv("doc_topic_weights.csv", encoding="utf-8")
    dom = (long.sort_values(["id","weight"], ascending=[True, False])
               .groupby("id", as_index=False).head(1))
    topic0_ids = dom.loc[dom["topic"] == 0, "id"].astype(int).tolist()
    topic0 = df[df["id"].isin(topic0_ids)].copy()
    topic0["weight"] = topic0["id"].map(dict(zip(dom["id"], dom["weight"])))

def excerpt(text, n=2):
    lines = [ln for ln in str(text).splitlines() if ln.strip()]
    return " / ".join(lines[:n])

topic0["excerpt"] = topic0["text"].map(lambda t: excerpt(t, 2))
topic0 = topic0.sort_values("weight", ascending=False).reset_index(drop=True)

topic0.to_csv("topic0_poems.csv", index=False, encoding="utf-8")
print(f"Poems assigned to Topic 0: {len(topic0)} -> topic0_poems.csv")
topic0.head(10)


Poems assigned to Topic 0: 26 -> topic0_poems.csv


,id,title,text,weight,excerpt
0,115,"I never lost as much but twice,","I never lost as much but twice,\nAnd that was ...",0.997753,"I never lost as much but twice, / And that was..."
1,198,THE SLEEPING FLOWERS.,"\n""Whose are the little beds,"" I asked,\n""Whic...",0.988396,"""Whose are the little beds,"" I asked, / ""Which..."
2,39,RENUNCIATION.,\nFirst page of Renunciation Second page of Re...,0.987938,First page of Renunciation Second page of Renu...
3,177,THE LETTER.,"\n""Going to him! Happy letter! Tell him â\nT...",0.987268,"""Going to him! Happy letter! Tell him â / Te..."
4,197,APRIL.,\nAn altered look about the hills;\nA Tyrian l...,0.979629,An altered look about the hills; / A Tyrian li...
5,109,"The daisy follows soft the sun,","The daisy follows soft the sun,\nAnd when his ...",0.978174,"The daisy follows soft the sun, / And when his..."
6,137,THE RETURN.,"\nThough I get home how late, how late!\nSo I ...",0.975877,"Though I get home how late, how late! / So I g..."
7,41,RESURRECTION.,"\n'T was a long parting, but the time\nFor int...",0.974536,"'T was a long parting, but the time / For inte..."
8,76,"One dignity delays for all,","One dignity delays for all,\nOne mitred aftern...",0.973809,"One dignity delays for all, / One mitred after..."
9,337,THANKSGIVING DAY.,\nOne day is there of the series\n Termed Th...,0.972222,One day is there of the series / Termed Tha...


In [15]:
# --- Top words (and exclusives) for Topic 0 ---

# Prefer in-memory H (topic×term) + vocab
if "H" in globals() and "vocab" in globals():
    k = 0
    idx = H[k].argsort()[::-1][:25]
    top_words_0 = [vocab[i] for i in idx]
else:
    # Fallback: parse topics_top_words.csv saved in Task 5
    tt = pd.read_csv("topics_top_words.csv", encoding="utf-8")
    words = tt.loc[tt["topic"] == 0, "top_words"].values
    top_words_0 = (words[0].split(", ") if len(words) else [])

print("Top words for Topic 0:")
print(", ".join(top_words_0[:20]))

# Optional: exclusivity using p(k|w) if H+W are in memory
exclusive_0 = []
if "H" in globals() and "W" in globals():
    H_pos = H + 1e-12
    pi_k = W.mean(axis=0)
    p_kw = (H_pos.T * pi_k).T
    p_kw = p_kw / (p_kw.sum(axis=0, keepdims=True) + 1e-12)
    k = 0
    excl_idx = np.argsort(p_kw[k])[::-1][:20]
    exclusive_0 = [vocab[i] for i in excl_idx]

    print("\nMost exclusive words for Topic 0:")
    print(", ".join(exclusive_0[:20]))


Top words for Topic 0:
'', one, tell, emily, dickinson, emily dickinson, poems, verses, first, know, time, letter, mine, upon, friends, day, page, series, thought, found

Most exclusive words for Topic 0:
emily dickinson, dickinson, emily, poems, verses, series, volume, beds, clearly, todd, mabel loomis, mabel, loomis todd, loomis, written, happy letter, get home, arrow, belshazzar, lines


In [16]:
# --- Top words (and exclusives) for Topic 0 ---

# Prefer in-memory H (topic×term) + vocab
if "H" in globals() and "vocab" in globals():
    k = 0
    idx = H[k].argsort()[::-1][:25]
    top_words_0 = [vocab[i] for i in idx]
else:
    # Fallback: parse topics_top_words.csv saved in Task 5
    tt = pd.read_csv("topics_top_words.csv", encoding="utf-8")
    words = tt.loc[tt["topic"] == 0, "top_words"].values
    top_words_0 = (words[0].split(", ") if len(words) else [])

print("Top words for Topic 0:")
print(", ".join(top_words_0[:20]))

# Optional: exclusivity using p(k|w) if H+W are in memory
exclusive_0 = []
if "H" in globals() and "W" in globals():
    H_pos = H + 1e-12
    pi_k = W.mean(axis=0)
    p_kw = (H_pos.T * pi_k).T
    p_kw = p_kw / (p_kw.sum(axis=0, keepdims=True) + 1e-12)
    k = 0
    excl_idx = np.argsort(p_kw[k])[::-1][:20]
    exclusive_0 = [vocab[i] for i in excl_idx]

    print("\nMost exclusive words for Topic 0:")
    print(", ".join(exclusive_0[:20]))


Top words for Topic 0:
'', one, tell, emily, dickinson, emily dickinson, poems, verses, first, know, time, letter, mine, upon, friends, day, page, series, thought, found

Most exclusive words for Topic 0:
emily dickinson, dickinson, emily, poems, verses, series, volume, beds, clearly, todd, mabel loomis, mabel, loomis todd, loomis, written, happy letter, get home, arrow, belshazzar, lines


In [17]:
# --- Suggest labels using motif lexicons + evidence from poems ---
import re
from collections import Counter

# Minimal motif lexicon (tweak freely)
LABEL_LEX = {
    "Death & Immortality": {"death","die","died","dead","funeral","grave","tomb","immortal","immortality","coffin","mourning"},
    "Nature: Bees & Flowers": {"bee","bees","hive","nectar","flower","flowers","bloom","blossom","rose","daisy","garden","buzz"},
    "Faith / Heaven": {"heaven","god","divine","angel","angels","prayer","soul","eternity","eternal","immortality"},
    "Time / Eternity": {"time","hour","hours","clock","noon","midnight","minute","eternity","forever","century","moment"},
    "Sea / Ship / Storm": {"sea","ocean","ship","boat","sail","sails","sailor","wave","waves","storm","shore","port"},
    "Light / Dark": {"light","sun","day","noon","dawn","bright","star","moon","night","dark","shade","shadow"},
    "Pain / Sorrow": {"pain","sorrow","grief","anguish","wound","wounds","aching","hurt","suffer","tears"},
    "Birds / Flight": {"bird","birds","robin","wing","wings","feather","feathers","fly","flight","sing","song"},
    "Home / Door / Room": {"house","home","room","rooms","door","doors","key","lock","window","chamber"},
    "Snake / Nature Frisson": {"snake","serpent","narrow","fellow","grass","coil","stripe"},
}

WORD_RE = re.compile(r"[A-Za-z’']+")

# 1) Build a bag-of-words for all Topic-0 poems
all_words = []
for txt in topic0["text"]:
    toks = [t.lower() for t in WORD_RE.findall(str(txt))]
    all_words.extend(toks)
bow = Counter(all_words)
N = sum(bow.values()) or 1

# 2) Compare Topic-0 top words against each lexicon
tw = set(w.lower() for w in (top_words_0 or []))
def jaccard(a, b): 
    return (len(a & b) / max(1, len(a | b)))

rows = []
for label, lex in LABEL_LEX.items():
    # overlap with Top words
    top_overlap = jaccard(tw, lex) if tw else 0.0
    # frequency evidence inside Topic-0 poems (per 1K tokens)
    freq = sum(bow[w] for w in lex)
    per_k = (1000.0 * freq / N)
    # simple combined score
    score = 0.6 * top_overlap + 0.4 * (per_k / (per_k + 20))  # saturating
    rows.append({"candidate_label": label, "overlap_topwords": round(top_overlap,3), 
                 "freq_per_1k": round(per_k,1), "score": round(score,3)})

suggestions = pd.DataFrame(rows).sort_values("score", ascending=False).reset_index(drop=True)
print("Label suggestions for Topic 0 (higher score = better):")
display(suggestions.head(5))

# Show which words matched (evidence)
def matches_for(label):
    lex = LABEL_LEX[label]
    top_hits = sorted((tw & lex))
    freq_hits = sorted([w for w in lex if bow[w] > 0], key=lambda w: -bow[w])
    return top_hits, [(w, bow[w]) for w in freq_hits[:12]]

best = suggestions.iloc[0]["candidate_label"]
th, fh = matches_for(best)
print(f"Top suggestion → {best}\nTop-word hits: {th}\nFrequent in Topic-0 poems: {fh[:8]}")


Label suggestions for Topic 0 (higher score = better):


,candidate_label,overlap_topwords,freq_per_1k,score
0,Light / Dark,0.057,6.7,0.134
1,Faith / Heaven,0.061,6.3,0.133
2,Time / Eternity,0.029,4.3,0.088
3,Nature: Bees & Flowers,0.000,5.0,0.080
4,Death & Immortality,0.000,4.3,0.071


Top suggestion → Light / Dark
Top-word hits: ['day', 'sun']
Frequent in Topic-0 poems: [('day', 8), ('sun', 5), ('night', 3), ('light', 2), ('dark', 1), ('dawn', 1)]


In [18]:
# Save Topic-0 label suggestions produced earlier as `suggestions`
from pathlib import Path

if 'suggestions' not in globals():
    raise NameError("Run the suggestions cell first to create `suggestions`.")

OUT_SUG_CSV  = Path("topic0_label_suggestions.csv")
OUT_SUG_JSON = Path("topic0_label_suggestions.json")

suggestions.to_csv(OUT_SUG_CSV, index=False, encoding="utf-8", lineterminator="\n")
suggestions.to_json(OUT_SUG_JSON, orient="records", force_ascii=False, indent=2)

print("Saved:")
print(" -", OUT_SUG_CSV.resolve())
print(" -", OUT_SUG_JSON.resolve())


Saved:
 - C:\Users\dkill\OneDrive\Documents\Poetry\Emily Dickinson\topic0_label_suggestions.csv
 - C:\Users\dkill\OneDrive\Documents\Poetry\Emily Dickinson\topic0_label_suggestions.json


### Keyword in context

In [5]:
# --- KWIC exporters ---
import pandas as pd

KWIC_TERMS  = ["death","immortality","bee","soul","heaven","time","flower","sea"]  # edit as needed
KWIC_WINDOW = 6  # tokens left/right

def kwic_for_term(term: str, docs_tok, ids, titles, window=6):
    term_lc = term.lower()
    rows = []
    for toks, pid, title in zip(docs_tok, ids, titles):
        for i, tok in enumerate(toks):
            if tok == term_lc:
                left  = " ".join(toks[max(0, i-window):i])
                right = " ".join(toks[i+1:i+1+window])
                rows.append({"id": int(pid), "title": str(title), "term": term_lc, "left": left, "keyword": tok, "right": right})
    return pd.DataFrame(rows)

for term in KWIC_TERMS:
    out = kwic_for_term(term, docs_tokens, df["id"], df["title"], window=KWIC_WINDOW)
    out.to_csv(f"kwic_{term}.csv", index=False, encoding="utf-8", lineterminator="\n", quoting=csv.QUOTE_MINIMAL)
    print(f"Saved kwic_{term}.csv (rows={len(out)})")


Saved kwic_death.csv (rows=51)
Saved kwic_immortality.csv (rows=11)
Saved kwic_bee.csv (rows=34)
Saved kwic_soul.csv (rows=40)
Saved kwic_heaven.csv (rows=54)
Saved kwic_time.csv (rows=60)
Saved kwic_flower.csv (rows=23)
Saved kwic_sea.csv (rows=40)


### Co-location network

In [6]:
# --- Collocation/PMI network ---
import math
import pandas as pd
from collections import Counter, defaultdict

CO_WINDOW   = 5   # lookahead window
MIN_COCOUNT = 4   # min pair count to keep
TOP_NODES   = None  # e.g., 300 to cap vocab

# token frequencies
token_counts = Counter()
for toks in docs_tokens:
    token_counts.update(toks)

# optional vocab cap
if TOP_NODES is not None and TOP_NODES < len(token_counts):
    allowed = set([w for w, _ in token_counts.most_common(TOP_NODES)])
else:
    allowed = set(token_counts.keys())

# co-occurrence within sliding window
def window_pairs(seq, win=CO_WINDOW):
    for i in range(len(seq)):
        base = seq[i]
        if base not in allowed: 
            continue
        seen = set()
        for w in seq[i+1:i+win]:
            if w in allowed and w != base:
                a, b = sorted((base, w))
                if (a, b) not in seen:
                    seen.add((a, b))
                    yield (a, b)

pair_counts = Counter()
for toks in docs_tokens:
    pair_counts.update(window_pairs(toks, win=CO_WINDOW))

# PMI computation
N_tokens   = sum(token_counts[w] for w in allowed)
total_pairs = sum(pair_counts.values())

edge_rows = []
for (a, b), cab in pair_counts.items():
    if cab < MIN_COCOUNT: 
        continue
    p_ab = cab / total_pairs if total_pairs else 0
    p_a  = token_counts[a] / N_tokens if N_tokens else 0
    p_b  = token_counts[b] / N_tokens if N_tokens else 0
    if p_ab > 0 and p_a > 0 and p_b > 0:
        pmi = math.log2(p_ab / (p_a * p_b))
        edge_rows.append({"source": a, "target": b, "co_count": int(cab), "PMI": round(pmi, 4)})

edges_df = pd.DataFrame(edge_rows).sort_values(["PMI","co_count"], ascending=[False, False])
edges_df.to_csv("collocations_edges.csv", index=False, encoding="utf-8", lineterminator="\n", quoting=csv.QUOTE_MINIMAL)

# node table
deg = defaultdict(int)
for s, t in zip(edges_df["source"], edges_df["target"]):
    deg[s] += 1; deg[t] += 1

nodes_df = pd.DataFrame(
    [{"term": w, "freq": int(token_counts[w]), "degree": int(deg.get(w, 0))} for w in allowed]
).sort_values(["degree","freq"], ascending=False)
nodes_df.to_csv("collocations_nodes.csv", index=False, encoding="utf-8", lineterminator="\n", quoting=csv.QUOTE_MINIMAL)

print(f"Saved collocations_edges.csv (edges={len(edges_df)}), collocations_nodes.csv (nodes={len(nodes_df)})")


Saved collocations_edges.csv (edges=278), collocations_nodes.csv (nodes=5279)


In [8]:
# --- Quick peeks ---
print("Topics sample:")
display(topics_df.head(10))

Topics sample:


,topic,top_words,weights
0,0,"'', one, tell, emily, dickinson, emily dickins...","37.06, 18.00, 12.28, 12.08, 12.08, 12.08, 10.0..."
1,1,"little, life, like, see, one, would, sun, morn...","18.17, 17.58, 12.00, 11.80, 11.65, 10.16, 9.66..."
2,2,"could, one, see, thee, day, new, would, sing, ...","7.87, 7.67, 6.87, 5.69, 5.51, 5.23, 5.14, 4.81..."
3,3,"like, tell, many, bee, done, rose, must, away,...","13.26, 9.19, 7.70, 6.42, 6.33, 5.97, 5.90, 5.6..."
4,4,"little, day, like, know, till, us, could, one,...","22.73, 22.61, 20.75, 18.69, 17.37, 13.06, 11.5..."
5,5,"little, one, away, like, heaven, never, could,...","22.65, 17.64, 15.19, 13.75, 12.89, 12.75, 11.5..."
6,6,"like, summer, know, one, still, god, death, ne...","18.04, 13.33, 12.71, 10.06, 9.47, 8.97, 8.53, ..."
7,7,"never, upon, go, except, could, heaven, must, ...","13.41, 9.85, 9.78, 9.26, 9.22, 8.56, 7.97, 7.5..."
8,8,"upon, like, one, could, '', look, still, know,...","20.21, 19.33, 13.89, 10.32, 9.00, 8.95, 8.33, ..."
9,9,"like, one, little, summer, upon, sun, could, b...","16.13, 15.86, 12.60, 12.27, 11.23, 10.40, 10.2..."


In [9]:
print("\nDoc-topic sample (top topic per poem):")
display(doc_topic_df.sort_values(["id","weight"], ascending=[True, False]).groupby("id").head(1).head(10))


Doc-topic sample (top topic per poem):


,id,title,topic,weight
4,1,SUCCESS.,4,0.963332
13,2,"Our share of night to bear,",1,0.946078
29,3,ROUGE ET NOIR.,5,0.938888
47,4,ROUGE GAGNE.,11,0.983024
51,5,Glee! The great storm is over!,3,0.973038
68,6,"If I can stop one heart from breaking,",8,0.963332
77,7,ALMOST!,5,0.961805
92,8,"A wounded deer leaps highest,",8,0.968390
97,9,"The heart asks pleasure first,",1,0.938887
114,10,IN A LIBRARY.,6,0.980902


In [10]:
print("\nTop PMI edges:")
display(edges_df.head(10))



Top PMI edges:


,source,target,co_count,PMI
256,forgetting,recollecting,4,10.9500
67,page,renunciation,7,10.1199
210,nights,wild,8,9.6281
274,adrift,boat,4,9.3126
156,dickinson,emily,12,8.5350
277,famous,sleep,4,8.2131
88,bees,murmuring,4,8.1199
239,dropped,flakes,4,8.1199
84,keep,sabbath,4,8.0325
35,election,mine,4,7.9907
